# Not Corgi — training on a Colab runtime

This notebook is a **driver**, not a second copy of the training code. It clones the
repo onto the Colab VM and runs `model/src/*.py` unchanged, so there is exactly one
implementation of data prep, training and evaluation and it is the one under version
control. If you find yourself pasting model code into a cell here, that is drift —
put it in `model/src/` and re-run the clone cell instead.

**Run the cells top to bottom.** Each one is idempotent: re-running after a runtime
disconnect picks up where you left off rather than starting over.

Two facts drive the whole layout below:

1. **The Colab VM is disposable.** It is wiped on disconnect and after ~90 minutes
   idle. Anything you want to keep — trained weights, metrics, figures — has to land
   in Google Drive, not on the VM.
2. **Google Drive is slow for many small files.** The Drive mount is a FUSE
   filesystem; reading thousands of individual JPEGs through it will starve the GPU
   and your epochs will be I/O-bound rather than compute-bound. So the dataset lives
   in Drive as *one archive*, gets copied to the VM's local SSD once, and training
   reads from there.

Setup instructions, including how to build that archive and how to drive this from
VS Code, are in [`model/colab/README.md`](./README.md).

## 1. Configuration

Everything the rest of the notebook needs is set here. Nothing below hardcodes a path.

In [ ]:
REPO_URL = "https://github.com/drateberry/not-corgi.git"

# Fresh clone, on the VM's local disk. Disposable — never edit code here and
# expect it to survive.
REPO_DIR = "/content/not-corgi"

# Persistent. Survives the runtime being recycled. Holds the dataset archive on
# the way in, and the trained model + reports on the way out.
DRIVE_ROOT = "/content/drive/MyDrive/not-corgi"

# Local SSD scratch. Fast. Wiped with the VM, which is fine — everything here is
# either derived from the archive or already copied to Drive.
WORK_DIR = "/content/notcorgi-work"

# The dataset archive you keep in Drive. See model/colab/README.md for how to
# assemble it. It should unzip to Cardigan/ Not_Corgi/ Pembroke/ folders of raw,
# unsplit source images — the same thing model/data/raw/ holds locally.
RAW_ARCHIVE = f"{DRIVE_ROOT}/data/raw.zip"

print(f"repo   {REPO_URL}\n  ->   {REPO_DIR}")
print(f"drive  {DRIVE_ROOT}")
print(f"scratch{WORK_DIR}")

## 2. Confirm you actually got a GPU

Worth thirty seconds up front. A Colab runtime with no accelerator attached trains
perfectly happily — just at roughly MacBook Air speed, which defeats the point of
being here. If this prints no GPU, fix the runtime type before going further
(in VS Code: re-select the kernel and pick a GPU runtime; in the browser:
*Runtime → Change runtime type*).

In [ ]:
import subprocess

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if smi.returncode == 0:
    print(smi.stdout)
else:
    print("No NVIDIA GPU attached to this runtime.")
    print("Change the runtime type to a GPU runtime, then re-run from the top.")

## 3. Mount Drive

Creates the folder layout on first run. You will be prompted to authorize; in VS Code
the prompt appears as a cell input box rather than a popup.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

for sub in ("data", "artifacts", "reports"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

print(f"{DRIVE_ROOT} contents:")
for entry in sorted(os.listdir(DRIVE_ROOT)):
    print(f"  {entry}")

if not os.path.exists(RAW_ARCHIVE):
    print(
        f"\nNOTE: {RAW_ARCHIVE} does not exist yet.\n"
        "Cells 1-5 will still run. Cell 6 (staging) needs it.\n"
        "See model/colab/README.md for how to build it."
    )

## 4. Get the code

Clones on first run, fast-forwards on later runs. This is the sync mechanism between
your Mac and the runtime: **commit and push locally, then re-run this cell.** The
Colab kernel cannot see your local filesystem, so an unpushed edit does not exist as
far as training is concerned — that is the single most common way to spend twenty
minutes debugging a change that was never actually running.

In [ ]:
import os
import subprocess


def git(*args):
    """Run a git command against the clone and stream its output."""
    result = subprocess.run(
        ["git", "-C", REPO_DIR, *args], capture_output=True, text=True
    )
    print((result.stdout + result.stderr).strip())
    return result


if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # --ff-only so a surprise merge conflict fails loudly here rather than
    # producing a half-merged working tree that then trains.
    git("fetch", "origin")
    git("pull", "--ff-only")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

print("\nNow at:")
git("log", "-1", "--oneline")
git("status", "--short")

## 5. Dependencies, and verify the environment

`requirements-colab.txt` installs only what a stock Colab image lacks — see the
comments in that file for why installing the full `requirements.txt` here is a bad
idea.

The verification cell then prints the exact versions you are training against, and
checks the one thing that fails silently: whether this Keras version bakes rescaling
into MobileNetV3. That is specifications.md §5.3. It reports the fact; acting on it
is `train.py`'s job.

In [ ]:
!pip install -q -r {REPO_DIR}/model/requirements-colab.txt

In [ ]:
import sys

import numpy as np
import tensorflow as tf

print(f"python      {sys.version.split()[0]}")
print(f"tensorflow  {tf.__version__}")
print(f"keras       {tf.keras.__version__}")
print(f"numpy       {np.__version__}")

gpus = tf.config.list_physical_devices("GPU")
print(f"\nGPUs visible to TensorFlow: {gpus or 'NONE — training will run on CPU'}")

# specifications.md 5.3 diagnostic. If preprocessing is baked into the backbone,
# calling preprocess_input on top of it double-normalizes and quietly destroys
# accuracy with no error message. Which of the two applies depends on the
# TensorFlow version this runtime happened to give you, so check rather than
# assume, and record the answer alongside the versions above.
try:
    probe = tf.keras.applications.MobileNetV3Small(weights=None, include_top=False)
    layer_types = {layer.__class__.__name__ for layer in probe.layers}
    baked_in = bool(layer_types & {"Rescaling", "Normalization"})
    print(
        f"\nMobileNetV3 bakes preprocessing into the graph: {baked_in}\n"
        + (
            "  -> Do NOT call preprocess_input. Feed raw [0, 255] pixels."
            if baked_in
            else "  -> This build does not rescale internally; preprocessing is on you."
        )
    )
    del probe
except Exception as exc:  # noqa: BLE001 — diagnostic only, never fail the run
    print(f"\nCould not probe MobileNetV3: {exc}")

## 6. Stage the dataset and wire up the output paths

This is the cell that makes the repo's scripts work unmodified on Colab. It replaces
four directories inside the clone with symlinks:

| Repo path | Points at | Why |
|---|---|---|
| `model/data/raw/` | local SSD | unzipped once from the Drive archive |
| `model/data/processed/` | local SSD | written by `prepare_data.py`, read every epoch — must be fast |
| `model/artifacts/` | Drive | the trained model; must survive the VM |
| `model/reports/` | Drive | confusion matrix and figures for `DESIGN.md` |

So `src/train.py` writes to `model/artifacts/` exactly as it does on your Mac, and the
weights land in Drive. No environment variables, no Colab-specific branches in the
training code, nothing to remember to undo before committing.

The `link()` helper refuses to replace a directory that holds real files, so a
mis-run cannot delete a dataset.

In [ ]:
import os
import shutil
import subprocess


def link(repo_relative_path, target):
    """Replace REPO_DIR/<repo_relative_path> with a symlink to <target>.

    Idempotent, and refuses to destroy anything that is not either absent or an
    empty placeholder directory (the repo ships these dirs containing only a
    .gitkeep, so that the expected layout is visible in a fresh clone).
    """
    source = os.path.join(REPO_DIR, repo_relative_path)
    os.makedirs(target, exist_ok=True)

    if os.path.islink(source):
        os.unlink(source)
    elif os.path.isdir(source):
        leftovers = [name for name in os.listdir(source) if name != ".gitkeep"]
        if leftovers:
            raise RuntimeError(
                f"Refusing to replace {source}: it contains {leftovers}. "
                "Move that content aside first."
            )
        shutil.rmtree(source)

    os.symlink(target, source)
    print(f"{repo_relative_path:24} -> {target}")


# --- unpack the dataset onto local SSD (skipped if already unpacked) ---------
raw_local = f"{WORK_DIR}/raw"
if os.path.isdir(raw_local) and os.listdir(raw_local):
    print(f"raw dataset already unpacked at {raw_local}")
else:
    if not os.path.exists(RAW_ARCHIVE):
        raise FileNotFoundError(
            f"{RAW_ARCHIVE} not found. See model/colab/README.md, 'Getting the "
            "dataset into Drive'."
        )
    os.makedirs(raw_local, exist_ok=True)
    # Copy off Drive first: unzipping straight from the FUSE mount is markedly
    # slower and more failure-prone than copying one large file and unzipping it
    # locally.
    staged = f"{WORK_DIR}/raw.zip"
    print(f"copying {RAW_ARCHIVE} -> {staged} ...")
    shutil.copyfile(RAW_ARCHIVE, staged)
    print("unzipping ...")
    subprocess.run(["unzip", "-q", staged, "-d", raw_local], check=True)
    os.remove(staged)

# --- point the repo at the right places -------------------------------------
print()
link("model/data/raw", raw_local)
link("model/data/processed", f"{WORK_DIR}/processed")
link("model/artifacts", f"{DRIVE_ROOT}/artifacts")
link("model/reports", f"{DRIVE_ROOT}/reports")

print("\nraw class folders found:")
for entry in sorted(os.listdir(raw_local)):
    path = os.path.join(raw_local, entry)
    count = len(os.listdir(path)) if os.path.isdir(path) else "-"
    print(f"  {entry:16} {count} files")

## 7. Prepare the data

Runs the repo's `src/prepare_data.py` with the working directory set to `model/`, which
is how `model/README.md` documents invoking it locally. Same command, same relative
paths, different machine.

Output is teed into `model/reports/` — which is Drive — so if the runtime drops
mid-run the log survives and you can see how far it got.

In [ ]:
%cd {REPO_DIR}/model
!python src/prepare_data.py 2>&1 | tee -a reports/prepare_data.log

## 8. Train

The long one. Notes before you start it:

- **Checkpoint from inside `train.py`, not from here.** A Colab runtime can be
  reclaimed mid-epoch. If the only copy of the weights is in memory when that
  happens, the run is gone. A `ModelCheckpoint` writing into `model/artifacts/`
  (which is Drive) turns a disconnect into an inconvenience instead of a restart.
- **Idle runtimes disconnect after roughly 90 minutes.** Actively training counts as
  activity; a finished cell with you away from the keyboard does not.
- Watch the first epoch's step time. If the GPU is barely utilised and step time is
  dominated by data loading, the dataset pipeline needs `cache()` / `prefetch()` —
  not a bigger GPU.

In [ ]:
%cd {REPO_DIR}/model
!python src/train.py 2>&1 | tee -a reports/train.log

## 9. Evaluate

Held-out test set only. Per specifications.md §6, read the confusion matrix and the
per-class precision/recall, not the aggregate accuracy — and treat a Pembroke vs.
Cardigan number at or near 99% as a leakage signal to investigate, not a result to
report.

In [ ]:
%cd {REPO_DIR}/model
!python src/evaluate.py 2>&1 | tee -a reports/evaluate.log

## 10. Confirm what actually made it to Drive

Run this before closing the runtime. `model/artifacts/` and `model/reports/` are
symlinks into Drive, so anything listed here is already persisted — but Drive syncs
asynchronously, and it is worth seeing non-zero file sizes with your own eyes rather
than discovering an empty `.keras` file tomorrow.

In [ ]:
import os

for folder in (f"{DRIVE_ROOT}/artifacts", f"{DRIVE_ROOT}/reports"):
    print(f"\n{folder}")
    entries = sorted(os.listdir(folder))
    if not entries:
        print("  (empty)")
    for name in entries:
        path = os.path.join(folder, name)
        size = os.path.getsize(path) if os.path.isfile(path) else 0
        print(f"  {name:40} {size / 1e6:8.2f} MB")

## 11. Bringing the results back to your Mac

The trained model is now in Drive under `not-corgi/artifacts/`. To point the Flask API
at it, download it into `api/artifacts/` locally — either through the Drive web UI, or
via the Google Drive folder on your Mac if you have the desktop client.

Two things to keep straight when you do:

- `model/artifacts/`, `api/artifacts/` and `model/data/` are gitignored (the
  submission ZIP has to stay under ~15MB), so the weights travel by Drive, not by git.
- `model/reports/` is **not** gitignored, and its figures are what `DESIGN.md` cites.
  Download those and commit them.

Inference stays on CPU per specifications.md §4.2 — nothing about training on a GPU
changes that, but it does mean you should sanity-check that the saved model loads and
predicts on your Mac before assuming the deployment path works.